# Sint lower-back 6-DoF adapter audit

This is an evidence-only audit before materialising or training a lower-back acceleration-plus-gyroscope model. It verifies the release-defined `lumbar` mapping, the six required channels, packet alignment with both feet, completeness, and sampling-rate exceptions. No classifier, normaliser, threshold, or frozen cohort is used.

**Decision rule:** materialisation is permitted only if every mapped trial has a release-defined lumbar file, all six channels, finite signals, sufficient packet overlap with both feet, and any non-100 Hz acquisition is explicitly resampled rather than silently treated as 100 Hz.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW = PROJECT_ROOT / 'data/raw/sint_maartenskliniek/extracted/IMU_GaitAnalysis-1.1.0/data'
PROCESSED = PROJECT_ROOT / 'data/processed'
INTERIM = PROJECT_ROOT / 'data/interim'
MANIFEST = pd.read_csv(PROCESSED / 'sint_maartenskliniek_trial_manifest.csv')
MAPPED = MANIFEST.loc[MANIFEST.export_exists].copy().sort_values(['participant', 'exported_trial'])
REQUIRED = ['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyr_X', 'Gyr_Y', 'Gyr_Z']
print(f'Mapped local trials: {len(MAPPED)} | participants: {MAPPED.participant.nunique()}')
display(MAPPED.groupby('label').agg(participants=('participant', 'nunique'), trials=('exported_trial', 'size')))

Mapped local trials: 79 | participants: 30


,participants,trials
label,,
healthy,20,59
stroke,10,20


In [2]:
def participant_folder(label):
    return RAW / ('CVA' if label == 'stroke' else 'Healthy_controls')

def read_export(path):
    lines = path.read_text(errors='replace').splitlines()
    header = next(i for i, line in enumerate(lines) if line.startswith('PacketCounter\t'))
    return pd.read_csv(path, sep='\t', skiprows=header, low_memory=False)

def sensor_paths(trial_dir):
    spec = json.loads((trial_dir / 'sensorspec.json.txt').read_text(encoding='utf-8'))
    paths = {}
    for location in ('lumbar', 'leftfoot', 'rightfoot'):
        suffix = spec[location]
        matches = list(trial_dir.glob(f'*_{suffix}'))
        if len(matches) != 1:
            raise RuntimeError(f'{location}: expected one match for {suffix}, found {len(matches)}')
        paths[location] = matches[0]
    return paths

rows = []
for record in MAPPED.itertuples(index=False):
    trial_dir = participant_folder(record.label) / record.participant / 'Xsens' / record.exported_trial
    row = {
        'dataset_id': 'sint_maartenskliniek', 'participant': record.participant,
        'label': record.label, 'trial': record.exported_trial,
        'vicon_trial': record.vicon_trial, 'trial_type': record.trial_type,
        'speed_condition': record.speed_condition,
        # The released importer identifies 900_V_01 *GRAIL* data as 40 Hz and resamples it to 100 Hz.
        'declared_source_fs_hz': 40 if (record.participant == '900_V_01' and record.trial_type != '2MWT') else 100,
        'requires_resample_to_100hz': (record.participant == '900_V_01' and record.trial_type != '2MWT'),
    }
    try:
        paths = sensor_paths(trial_dir)
        frames = {name: read_export(path) for name, path in paths.items()}
        row['release_defined_lumbar_mapping'] = paths['lumbar'].name.endswith('00B40A8D.txt')
        row['all_required_columns'] = all(set(REQUIRED).issubset(frame.columns) for frame in frames.values())
        numeric = {name: frame[REQUIRED].apply(pd.to_numeric, errors='coerce').to_numpy(float) for name, frame in frames.items()}
        counters = {name: pd.to_numeric(frame['PacketCounter'], errors='coerce').to_numpy(float) for name, frame in frames.items()}
        for name, values in numeric.items():
            row[f'{name}_rows'] = len(values)
            row[f'{name}_finite_fraction'] = float(np.isfinite(values).all(axis=1).mean())
        lb_acc = np.linalg.norm(numeric['lumbar'][:, :3], axis=1)
        lb_gyr = np.linalg.norm(numeric['lumbar'][:, 3:], axis=1)
        row['lumbar_acc_mag_median_ms2'] = float(np.nanmedian(lb_acc))
        row['lumbar_gyr_mag_median_native'] = float(np.nanmedian(lb_gyr))
        row['lumbar_acc_abs_max_ms2'] = float(np.nanmax(np.abs(numeric['lumbar'][:, :3])))
        row['lumbar_gyr_abs_max_native'] = float(np.nanmax(np.abs(numeric['lumbar'][:, 3:])))
        lb_counter = counters['lumbar'][np.isfinite(counters['lumbar'])]
        row['lumbar_packet_gap_fraction'] = float((np.diff(lb_counter) != 1).mean()) if len(lb_counter) > 1 else 1.0
        for foot in ('leftfoot', 'rightfoot'):
            common = np.intersect1d(lb_counter, counters[foot][np.isfinite(counters[foot])])
            row[f'lumbar_{foot}_packet_overlap'] = float(len(common) / min(len(lb_counter), len(counters[foot])))
        row['status'] = 'audited'
    except Exception as exc:
        row['status'] = f'error:{type(exc).__name__}'
        row['error'] = str(exc)
    rows.append(row)

audit = pd.DataFrame(rows)
display(audit.groupby(['label', 'status']).size().rename('trials').reset_index())
display(audit[['participant', 'trial', 'declared_source_fs_hz', 'requires_resample_to_100hz', 'release_defined_lumbar_mapping', 'all_required_columns', 'lumbar_finite_fraction', 'lumbar_leftfoot_packet_overlap', 'lumbar_rightfoot_packet_overlap']].head())

,label,status,trials
0,healthy,audited,59
1,stroke,audited,20


,participant,trial,declared_source_fs_hz,requires_resample_to_100hz,release_defined_lumbar_mapping,all_required_columns,lumbar_finite_fraction,lumbar_leftfoot_packet_overlap,lumbar_rightfoot_packet_overlap
0,900_CVA_01,exported000,100,False,True,True,1.0,1.0,1.0
1,900_CVA_01,exported001,100,False,True,True,1.0,1.0,1.0
2,900_CVA_02,exported000,100,False,True,True,1.0,1.0,1.0
3,900_CVA_02,exported002,100,False,True,True,1.0,1.0,1.0
4,900_CVA_03,exported002,100,False,True,True,1.0,1.0,1.0


In [3]:
assert len(audit) == len(MAPPED), 'Every mapped local trial must be audited.'
assert audit.status.eq('audited').all(), audit.loc[~audit.status.eq('audited'), ['participant', 'trial', 'status', 'error']]
assert audit.release_defined_lumbar_mapping.all(), 'Lumbar mapping must come from every release sensorspec.'
assert audit.all_required_columns.all(), 'One or more exports lack a required 6-DoF field.'
for col in ['lumbar_finite_fraction', 'leftfoot_finite_fraction', 'rightfoot_finite_fraction', 'lumbar_leftfoot_packet_overlap', 'lumbar_rightfoot_packet_overlap']:
    assert (audit[col] >= 0.999).all(), f'Failed quality floor: {col}'
assert audit.loc[audit.requires_resample_to_100hz, 'participant'].eq('900_V_01').all()
assert audit.loc[audit.requires_resample_to_100hz, 'trial_type'].ne('2MWT').all()

INTERIM.mkdir(parents=True, exist_ok=True)
audit_path = INTERIM / 'sint_lower_back_6dof_adapter_audit.csv'
audit.to_csv(audit_path, index=False)
summary = {
    'mapped_trials': int(len(audit)),
    'participants': int(audit.participant.nunique()),
    'healthy_participants': int(audit.loc[audit.label.eq('healthy'), 'participant'].nunique()),
    'stroke_participants': int(audit.loc[audit.label.eq('stroke'), 'participant'].nunique()),
    'all_trials_passed_6dof_mapping_and_quality': True,
    'rate_exception': '900_V_01 GRAIL trials only; resample each affected raw stream from 40 Hz to 100 Hz before windowing',
    'next_step': 'versioned lower-back acceleration-plus-gyroscope materialisation; no overwrite of the 3-channel magnitude tensor'
}
summary_path = INTERIM / 'sint_lower_back_6dof_adapter_audit_summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(json.dumps(summary, indent=2))
print(f'\nWrote: {audit_path.relative_to(PROJECT_ROOT)}')

{
  "mapped_trials": 79,
  "participants": 30,
  "healthy_participants": 20,
  "stroke_participants": 10,
  "all_trials_passed_6dof_mapping_and_quality": true,
  "rate_exception": "900_V_01 GRAIL trials only; resample each affected raw stream from 40 Hz to 100 Hz before windowing",
  "next_step": "versioned lower-back acceleration-plus-gyroscope materialisation; no overwrite of the 3-channel magnitude tensor"
}

Wrote: data\interim\sint_lower_back_6dof_adapter_audit.csv


In [4]:
# Compare only the already-standardised lower-back magnitude scales. Gyroscope scale is retained
# as native raw provenance here; later training must use fold-fitted source-aware normalisation.
fv_x = np.load(PROCESSED / 'validated_gait_windows_float32.npy', mmap_mode='r')
fv_m = pd.read_csv(PROCESSED / 'validated_window_metadata.csv')
fv = pd.DataFrame({
    'dataset_id': fv_m.dataset_id,
    'lb_acc_mag_g_median_per_window': np.median(np.linalg.norm(fv_x[:, :, :3], axis=2), axis=1),
    'lb_gyr_mag_native_median_per_window': np.median(np.linalg.norm(fv_x[:, :, 3:6], axis=2), axis=1),
})
sint_x = np.load(PROCESSED / 'sint_maartenskliniek_external_windows_float32.npy', mmap_mode='r')
sint_summary = pd.DataFrame({
    'dataset_id': ['sint_maartenskliniek'],
    'lb_acc_mag_g_median_per_window_mean': [float(np.median(sint_x[:, :, 0], axis=1).mean())],
    'lb_acc_mag_g_median_per_window_sd': [float(np.median(sint_x[:, :, 0], axis=1).std())],
    'lb_gyr_mag_native_median_per_window_mean': [np.nan],
    'lb_gyr_mag_native_median_per_window_sd': [np.nan],
})
fv_summary = fv.groupby('dataset_id').agg(
    lb_acc_mag_g_median_per_window_mean=('lb_acc_mag_g_median_per_window', 'mean'),
    lb_acc_mag_g_median_per_window_sd=('lb_acc_mag_g_median_per_window', 'std'),
    lb_gyr_mag_native_median_per_window_mean=('lb_gyr_mag_native_median_per_window', 'mean'),
    lb_gyr_mag_native_median_per_window_sd=('lb_gyr_mag_native_median_per_window', 'std'),
).reset_index()
comparison = pd.concat([fv_summary, sint_summary], ignore_index=True)
comparison.to_csv(INTERIM / 'lower_back_accel_gyro_scale_provenance_comparison.csv', index=False)
display(comparison.round(4))
print('Conclusion: Sint passes structural adaptation. The next notebook must materialise a versioned 2-channel LB tensor, resampling only the documented 900_V_01 exception, then test Felius/Voisard/Sint lower-back transport with fold-fitted normalisation.')

,dataset_id,lb_acc_mag_g_median_per_window_mean,lb_acc_mag_g_median_per_window_sd,lb_gyr_mag_native_median_per_window_mean,lb_gyr_mag_native_median_per_window_sd
0,felius_2024,0.9793,0.0263,28.9721,13.8118
1,voisard_2025,0.9988,0.0244,28.6398,7.8233
2,sint_maartenskliniek,1.0052,0.0336,NaN,NaN


Conclusion: Sint passes structural adaptation. The next notebook must materialise a versioned 2-channel LB tensor, resampling only the documented 900_V_01 exception, then test Felius/Voisard/Sint lower-back transport with fold-fitted normalisation.


## Audit outcome

Passing this audit establishes **structural compatibility**, not clinical equivalence or guaranteed model improvement. The next experiment must preserve participant-disjoint splits, fit every normalisation parameter on each training fold only, and keep RevalExo and the frozen cohorts out of all design choices.